In [9]:
from promenade.models import *
from qdrant_client.models import Filter, FieldCondition, MatchValue
QDRANT_PATH = DATA_DIR / "qdrant"
retrieve_reranker = RetreiveReranker()

In [10]:
TimeString = Annotated[str, "format: HH:MM:SS.ffffff"]


def validate_time_format(v: str) -> str:
    """Validate and normalize time format to HH:MM:SS.ffffff"""
    import re
    time = v
    
    # If just hour like "10" -> "10:00:00.000000"
    if re.match(r"^\d{1,2}$", v):
        time = f"{int(v):02d}:00:00.000000"
    
    # If HH:MM like "10:00" -> "10:00:00.000000"
    if re.match(r"^\d{1,2}:\d{2}$", v):
        parts = v.split(":")
        time = f"{int(parts[0]):02d}:{parts[1]}:00.000000"
    
    # Full format HH:MM:SS.ffffff - validate each part
    if re.match(r"^\d{1,2}:\d{2}:\d{2}\.\d+$", time):
        parts = v.split(":")
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds_part = parts[2].split(".")
        seconds = int(seconds_part[0])
        
        # Validate time ranges
        if hours > 23:
            raise ValueError(f"Invalid time: hours must be 0-23, got {hours}")
        if minutes > 59:
            raise ValueError(f"Invalid time: minutes must be 0-59, got {minutes}")
        if seconds > 59:
            raise ValueError(f"Invalid time: seconds must be 0-59, got {seconds}")
        
        return time
    
    # Invalid format - raise error
    raise ValueError(f"Invalid time format: {v}. Expected HH, HH:MM, or HH:MM:SS.ffffff")


class FilterState(TypedDict):
    input_query: str
    start_time: TimeString | None
    end_time: TimeString | None
    day_of_week: int | None
    filtred_places: list[tuple] | None

class TimeSpace(BaseModel):
    start_time: str
    end_time: str
    day_of_week: int

    @field_validator("start_time", "end_time", mode="before")
    @classmethod
    def validate_time(cls, v: str) -> str:
        return validate_time_format(v)

    @field_validator("day_of_week", mode="before")
    @classmethod
    def validate_day(cls, v: int) -> int:
        if v < 0 or v > 6:
            raise ValueError("day_of_week must be 0-6 (Monday-Sunday)")
        return v

In [11]:
from datetime import datetime
import calendar
today = datetime.today().strftime('%Y-%m-%d')
weekday_num = datetime.today().weekday()
day_name_full = calendar.day_name[weekday_num]

In [12]:
structured_llm = llm.with_structured_output(TimeSpace)
QUERY_TRANSFORM_SYSTEM = f"""You are a structured data extractor. 
Your only job is to transform user query into TimeSpace schema.
today is {today}, {day_name_full}.

User will ask about when they can visit places. Extract the time window they are interested in.
The day_of_week should be extracted from phrases like "в следующий вторник", "в четверг", "в субботу" etc.
Days of week mapping: Monday=0, Tuesday=1, Wednesday=2, Thursday=3, Friday=4, Saturday=5, Sunday=6.

IMPORTANT: Time format MUST be HH:MM:SS.ffffff (e.g., 10:00:00.000000, 21:00:00.000000).
If user gives time as "10" or "10:00", convert it to "10:00:00.000000".
If user gives time as "21:00", convert it to "21:00:00.000000".

Return ONLY valid JSON matching TimeSpace schema with keys: start_time, end_time, day_of_week.
"""

In [13]:
def tranform_user_query(state: FilterState):
    time_window = structured_llm.invoke(
        [
            SystemMessage(QUERY_TRANSFORM_SYSTEM), 
            HumanMessage(state["user_query"])
        ]
    )
    
    return {
        "start_time": time_window.start_time,
        "end_time": time_window.end_time,
        "day_of_week": time_window.day_of_week
    }

In [14]:
time_window = tranform_user_query({"user_query": "Куда я могу сходить в следующий вторник с 10 до 21"})

In [15]:
time_window

{'start_time': '10:00:00.000000',
 'end_time': '21:00:00.000000',
 'day_of_week': 1}

In [17]:
def filter_database(state: FilterState):
    
    
    QUERY = f"""
SELECT 
    s.museum_id, m.museum_name, m.url
FROM schedule s
LEFT JOIN museum m
    on m.id = s.museum_id
WHERE 
    is_closed IS 0
    AND day_of_week IS {state["day_of_week"]}
    AND open_time <= "{state["start_time"]}"
    AND CASE
        WHEN last_entry_time IS NOT NULL 
        THEN last_entry_time >= "{state["end_time"]}"
        ELSE close_time >= "{state["end_time"]}"
    END

"""
    engine = create_engine(DATABASE_ADRESS)
    with engine.connect() as conn:
        res = conn.execute(text(QUERY))
        rows = res.fetchall()
    return {
        "filtred_places": rows
    }

In [18]:
rows = filter_database(time_window)
rows

{'filtred_places': [(8, 'Скалодром RedPoint (Москва)', 'http://redpoint.msk.ru/menu/ratesandhours/'),
  (9, 'Главное здание', 'https://pushkinmuseum.art/'),
  (12, 'Галерея искусства стран Европы и Америки', 'https://pushkinmuseum.art/')]}